# Phase 4 on Kaggle: T4 check, LoRA leftovers, stacking 12-best, matrix scoring

Runs `pipeline/kaggle_phase4.py` (the code comes from the private Hugging Face repo `OmiKhan/bangla-asr-hub`,
together with the audio, the manifests and the models). Step-by-step guide: `docs/PHASE4_RUNBOOK.md`.

Before **Save Version -> Save & Run All (Commit)**:
1. Right panel -> *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. *Add-ons -> Secrets*: a secret named **HF_TOKEN** (the read-only Hugging Face token), switched on for this notebook.
3. In the next cell, choose the jobs for GPU 0 and GPU 1 (one preset per session).


In [ ]:
# 1. Which jobs this session runs (copy one preset; each GPU runs its list in order).
# Session A (T4 check + LoRA leftovers, ~1.5-2.5 h):
GPU0 = "t4check"
GPU1 = "lora"
# Session B (stacking input for M1 full; eval is the longest, ~5-7 h est.):
# GPU0 = "stack:eval"
# GPU1 = "stack:dev,stack:fleurs,stack:kathbath"
# Matrix model scoring later (model uploaded to the hub as models/<run>):
# GPU0 = "score:m2_cv"
# GPU1 = "score:m3_kathbath"
RESUME_FROM = ""   # to continue a session that stopped: add its output as input and paste the path of its phase4 folder here


In [ ]:
# 2. Install the same library versions as on Modal (Kaggle's own torch is kept; the T4 check verifies the numbers).
!pip install -q "transformers==5.15.0" "peft==0.21.2" "accelerate==1.14.0" "jiwer==4.0.0" "rapidfuzz==3.14.5"     "soundfile==0.14.0" "soxr==1.1.0" "librosa==1.0.0" "huggingface_hub>=1.5,<2" hf_xet


In [ ]:
# 3. Hugging Face token, GPU check, and the code of the commit recorded in the hub.
import os, subprocess, sys, tarfile, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
HUB = "OmiKhan/bangla-asr-hub"
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "No GPU: Session options -> Accelerator -> GPU T4 x2, then restart."
commit = open(hf_hub_download(HUB, "code/latest.txt", repo_type="dataset")).read().strip()
tarfile.open(hf_hub_download(HUB, f"code/{commit}.tar.gz", repo_type="dataset")).extractall("/tmp/code")
print("code commit", commit)


In [ ]:
# 4. Run the jobs (both GPUs at once). Progress lines every 5 minutes; logs in /kaggle/working/phase4/logs.
cmd = [sys.executable, "/tmp/code/pipeline/kaggle_phase4.py", "--gpu0", GPU0, "--gpu1", GPU1]
if RESUME_FROM:
    cmd += ["--resume_from", RESUME_FROM]
p = subprocess.run(cmd, env={**os.environ, "PYTHONPATH": "/tmp/code/src"})
print("exit code", p.returncode, "(0 = every job finished)")


In [ ]:
# 5. What to download: /kaggle/working/phase4_outputs.zip (Output tab). Quick look at the summary:
print(open("/kaggle/working/phase4/summary.json").read())
